# YOLO11n 교통약자 감지 파인튜닝
**대상 클래스**: Wheelchair, Crutch  
**데이터셋**: Open Images V7 (fiftyone 자동 다운로드)  
**목표**: 라즈베리파이 5 CPU 추론용 경량 모델

## 사전 준비
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 아래 셀을 순서대로 실행 (Drive 업로드 불필요 — 자동 다운로드)

In [ ]:
# 1. GPU 확인
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ GPU 없음 — 런타임 유형을 T4 GPU로 변경하세요')
print('CUDA:', torch.version.cuda)
!nvidia-smi | head -20

In [ ]:
# 2. 패키지 설치
!pip install ultralytics fiftyone -q
print('설치 완료')

In [ ]:
# 3. Open Images V7 데이터셋 다운로드 (fiftyone)
# Wheelchair + Crutch 클래스만 선택적 다운로드
import fiftyone as fo
import fiftyone.zoo as foz
import os

CLASSES   = ['Wheelchair', 'Crutch']
BASE_DIR  = '/content/datasets/mobility_aids'
SPLIT_MAP = {'train': 'train', 'validation': 'val'}

os.makedirs(BASE_DIR, exist_ok=True)

for split, out_name in SPLIT_MAP.items():
    print(f'\n[{split}] 다운로드 시작...')
    ds = foz.load_zoo_dataset(
        'open-images-v7',
        split=split,
        classes=CLASSES,
        label_types=['detections'],
        only_matching=True,
    )
    print(f'  → {len(ds)}장 다운로드 완료')

    ds.export(
        export_dir=f'{BASE_DIR}/{out_name}',
        dataset_type=fo.types.YOLOv5Dataset,
        label_field='ground_truth',
        classes=CLASSES,
    )
    print(f'  → YOLO 포맷 export 완료: {BASE_DIR}/{out_name}')
    fo.delete_dataset(ds.name)

print('\n✅ 전체 다운로드 완료')
!find /content/datasets/mobility_aids -name '*.jpg' | wc -l | xargs echo '총 이미지:'

In [ ]:
# 4. dataset.yaml 생성
import yaml, os

# fiftyone export 후 실제 이미지/라벨 경로 확인
def find_images_dir(base):
    for root, dirs, files in os.walk(base):
        if any(f.endswith(('.jpg','.jpeg','.png')) for f in files):
            return root
    return base

train_img = find_images_dir('/content/datasets/mobility_aids/train')
val_img   = find_images_dir('/content/datasets/mobility_aids/val')

# train_img, val_img 에서 BASE_DIR 기준 상대경로 추출
BASE_DIR = '/content/datasets/mobility_aids'
train_rel = os.path.relpath(train_img, BASE_DIR)
val_rel   = os.path.relpath(val_img,   BASE_DIR)

cfg = {
    'path' : BASE_DIR,
    'train': train_rel,
    'val'  : val_rel,
    'nc'   : 2,
    'names': {0: 'Wheelchair', 1: 'Crutch'},
}

yaml_path = f'{BASE_DIR}/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('dataset.yaml 생성 완료:')
!cat /content/datasets/mobility_aids/dataset.yaml
print(f'\ntrain 이미지: {train_img}')
print(f'val   이미지: {val_img}')
print(f'train 이미지 수:', len([f for f in os.listdir(train_img) if f.endswith(('.jpg','.png'))]))
print(f'val   이미지 수:', len([f for f in os.listdir(val_img)   if f.endswith(('.jpg','.png'))]))

In [ ]:
# 5. YOLO11n 파인튜닝 (T4 GPU, ~1시간)
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/datasets/mobility_aids/dataset.yaml',
    epochs=100,
    batch=32,
    imgsz=640,
    device=0,        # T4 GPU
    patience=20,     # early stopping
    workers=2,
    project='/content/runs',
    name='mobility_yolo11n',
    exist_ok=True,
)

print('\n✅ 학습 완료!')
print('저장 경로:', results.save_dir)

In [ ]:
# 6. 학습 결과 확인 및 검증
from ultralytics import YOLO

best = '/content/runs/mobility_yolo11n/weights/best.pt'
last = '/content/runs/mobility_yolo11n/weights/last.pt'

!ls -lh /content/runs/mobility_yolo11n/weights/

model_best = YOLO(best)
metrics = model_best.val(data='/content/datasets/mobility_aids/dataset.yaml')
print(f'\nmAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'Precision : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')

In [ ]:
# 7. Google Drive에 모델 저장 (선택)
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/sbms-pi/models'
os.makedirs(save_dir, exist_ok=True)

shutil.copy(best, f'{save_dir}/mobility_yolo11n_best.pt')
shutil.copy(last, f'{save_dir}/mobility_yolo11n_last.pt')

print(f'✅ 모델 저장 완료 → {save_dir}')
!ls -lh /content/drive/MyDrive/sbms-pi/models/

In [ ]:
# 8. (선택) Raspberry Pi용 OpenVINO INT8 export — FPS 5→20 향상
from ultralytics import YOLO

model = YOLO('/content/runs/mobility_yolo11n/weights/best.pt')
model.export(
    format='openvino',
    imgsz=640,
    int8=True,
)

import shutil, os
ov_dir = '/content/runs/mobility_yolo11n/weights/best_openvino_model'
save_dir = '/content/drive/MyDrive/sbms-pi/models'
shutil.copytree(ov_dir, f'{save_dir}/best_openvino_model', dirs_exist_ok=True)
print('✅ OpenVINO 모델 저장 완료')

## 완료 후 — sola-1 배포

### 1. 모델 다운로드
Drive에서 `sbms-pi/models/mobility_yolo11n_best.pt` 다운로드

### 2. sola-1으로 전송
```bash
scp mobility_yolo11n_best.pt sola-tunnel:/home/admin/gunpo/docker/cv/
```

### 3. cv_ffmpeg.py 모델 경로 확인
```bash
# docker/cv/cv_ffmpeg.py 에서 MODEL_PATH가 best.pt를 가리키는지 확인
ssh sola-tunnel 'ls -lh ~/gunpo/docker/cv/*.pt'
```

### 4. 서비스 재시작
```bash
ssh sola-tunnel 'sudo systemctl restart cv_ffmpeg'
```

### 5. 디버그 스트림 확인
브라우저에서 `http://sola-ip:8089/stream` 열어서 교통약자 감지 확인